# Cross-session L9 replay diagnostic

Target for 5 October 2026: locate the first recorded divergence between two fresh
Colab sessions learning task 9 from the same post-U3 checkpoint.

Use this notebook three times: session A on GPU, session B on a fresh GPU runtime,
then comparison on CPU. Both GPU runs use the same seed, restored PyTorch RNG,
code, settings, and data. Python and NumPy RNG are reset to the checkpoint seed
because the historical checkpoint does not save them. This tests reproducibility
of this replay policy; it does not establish that the historical policy matched.

Session A creates a contract on Drive. Session B checks it and never resumes A's
final model. Each label can run only once per experiment ID. If interrupted, keep
the partial artifacts and start a new experiment ID with a new A/B pair.

Per-update parameter hashing copies the model to CPU and adds runtime cost.
Keep this instrumentation identical in both sessions. It does not measure normal
training throughput. Do not enable new deterministic settings for the first pair;
record the defaults and apply the same settings in B.

The tracing code must be available in the Git revision selected below. This local
notebook and module must be committed and pushed before cloning them in Colab.


## 1. Freeze the replay configuration

Choose a unique `EXPERIMENT_ID`. Run A first. In a fresh session use the identical
ID and set `MODE = "B"`. For the final CPU session set `MODE = "COMPARE"`.
The checkpoint path is the one used by the existing paired L9 workflow; verify it
against your Drive. `REVISION` must name a revision containing this notebook's
support code. Session B and comparison automatically use A's full commit ID.


In [ ]:
from pathlib import Path
import json
from google.colab import drive

drive.mount("/content/drive")
MODE = "A"  # A, B, or COMPARE
EXPERIMENT_ID = "20261005_L9_replay_01"
REVISION = "main"  # A resolves and saves the full commit ID.
CHECKPOINT = Path("/content/drive/MyDrive/uncle/E08_forgetting_trace/full_sequence_27step_candidate/checkpoint_seq1_resnet50_seed0_after_request02_forget3.pt")
DATA = Path("/content/data/tiny-imagenet-200")
EXPERIMENT = Path("/content/drive/MyDrive/uncle/E08_forgetting_trace/L9_session_replay") / EXPERIMENT_ID
CONTRACT = EXPERIMENT / "contract.json"
assert MODE in {"A", "B", "COMPARE"}
if MODE != "A":
    assert CONTRACT.is_file(), "Run session A first."
    REVISION = json.loads(CONTRACT.read_text())["code"]["commit"]
else:
    assert not EXPERIMENT.exists(), "Use a new experiment ID; preserve prior artifacts."


In [ ]:
# Run before importing uncle. Safe to retry after a failed setup cell.
import os
import subprocess
import sys
REPO = Path("/content/uncle-L9-replay")
assert not any(name == "uncle" or name.startswith("uncle.") for name in sys.modules), "Restart the runtime before changing imported code."
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/sumitasthana/CARK.git", str(REPO)], check=True)
else:
    assert (REPO / ".git").is_dir(), "The setup path is not a Git checkout. Use a fresh runtime."
    remote = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip().removesuffix(".git")
    assert remote == "https://github.com/sumitasthana/CARK", "Unexpected repository at the setup path."
    status = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip()
    assert not status, "Preserve local changes and use a fresh runtime."
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
selected = "origin/main" if REVISION == "main" else REVISION
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", selected], check=True)
assert (REPO / "uncle/replay.py").is_file(), "Selected revision does not contain the replay workflow. Use a published revision containing notebook 11 and uncle/replay.py."
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


## 2. Run session A from post-U3

This cell runs only the selected GPU label. Session B uses the same cell after
starting a fresh runtime in step 4. The trace includes step zero after task-9
initialization, then every completed Adam update. Loss is measured before that
update; parameter and task-buffer hashes are measured after it. Starting task-9
accuracy is the newly initialized task's accuracy.

The sample indices in the trace map to stable image names, labels, and file hashes
in `data_manifest.json`. Starting evaluation is isolated from training RNG and
module modes. All previously seen tasks, including forgotten task 3, remain
protected as in the existing request runner.


In [ ]:
if MODE in {"A", "B"}:
    # Set the cuBLAS environment before importing or initializing CUDA.
    if MODE == "B":
        workspace = json.loads(CONTRACT.read_text())["settings"]["cublas_workspace_config"]
        if workspace is None:
            os.environ.pop("CUBLAS_WORKSPACE_CONFIG", None)
        else:
            os.environ["CUBLAS_WORKSPACE_CONFIG"] = workspace
    import torch
    from uncle import checkpoint as checkpointing
    from uncle.config import Config
    from uncle.streams import build_tasks
    from uncle.replay import run_replay

    assert torch.cuda.is_available(), "Select a GPU runtime."
    assert CHECKPOINT.is_file(), CHECKPOINT
    # Apply A's recorded backend flags before constructing the model in B.
    if MODE == "B":
        settings = json.loads(CONTRACT.read_text())["settings"]
        torch.use_deterministic_algorithms(settings["deterministic_algorithms"],
                                          warn_only=settings["deterministic_warn_only"])
        torch.backends.cudnn.benchmark = settings["cudnn_benchmark"]
        torch.backends.cudnn.deterministic = settings["cudnn_deterministic"]
        torch.set_float32_matmul_precision(settings["float32_matmul_precision"])
        torch.backends.cudnn.allow_tf32 = settings["cudnn_allow_tf32"]
        torch.backends.cuda.matmul.allow_tf32 = settings["matmul_allow_tf32"]
    payload = checkpointing.load(CHECKPOINT)
    config = Config(**payload["config"])
    del payload
    assert config.dataset == "tiny_imagenet" and config.backbone == "resnet50"
    tasks = build_tasks(config, root=DATA, include=("0", "3", "9"), download=True)
    EXPERIMENT.mkdir(parents=True, exist_ok=True)
    RUN_FOLDER = run_replay(CHECKPOINT, tasks, EXPERIMENT / MODE, CONTRACT, label=MODE)
    print("Completed:", RUN_FOLDER)


## 3. Verify Drive artifacts and release the GPU

Run this after either A or B. It verifies every saved artifact hash and the expected
number of updates before flushing Drive and releasing the runtime. The original
environment record is preserved in that label's folder. If verification fails,
inspect the error before releasing the runtime.


In [ ]:
if MODE in {"A", "B"}:
    from uncle.replay import verify_run
    from google.colab import runtime
    rows = verify_run(RUN_FOLDER)
    print("Verified", len(rows), "trace rows in", RUN_FOLDER, flush=True)
    drive.flush_and_unmount(timeout_ms=600000)
    print("Drive flushed. Releasing GPU.", flush=True)
    runtime.unassign()


## 4. Repeat in a fresh GPU session

After A releases its runtime, start a fresh GPU runtime. Set `MODE = "B"`, keep
`EXPERIMENT_ID` and `CHECKPOINT` unchanged, and run steps 1 through 3. Session B
checks the source checkpoint, full code revision, settings, and ordered data
manifest against A's contract. It writes only to the B folder and releases its GPU.

## 5. Compare the saved traces on CPU

Start a fresh CPU runtime, set `MODE = "COMPARE"`, and run step 1 followed by this
cell. No datasets or GPU are required for comparison. The report identifies
runtime metadata differences separately from the first trace mismatch. A match at
all recorded points means the historical spread was not reproduced by this pair;
it does not establish that its cause has been resolved.


In [ ]:
if MODE == "COMPARE":
    from uncle.replay import compare_replays
    report = compare_replays(EXPERIMENT / "A", EXPERIMENT / "B",
                             EXPERIMENT / "comparison.json")
    print(json.dumps(report, indent=2))
    assert json.loads((EXPERIMENT / "comparison.json").read_text()) == report
    drive.flush_and_unmount(timeout_ms=600000)


Interpret the earliest mismatch before choosing another run:

- Starting state or embedding: check restore and initialization.
- Ordered sample indices: check sampling and dataset indexing.
- Input hash with matching indices: check image bytes and preprocessing.
- First loss or parameter hash with matching state and inputs: check runtime
  versions, GPU, backend settings, and numerical operations.
- A later mismatch: inspect the preceding matching update and its RNG and buffer
  hashes. Matching hashes do not expose every possible source of nondeterminism.

These are investigation paths, not conclusions. After the replay difference is
explained, the next experiment is a learning-only sequence measuring older-task
accuracy after each learn request. Hold gamma tuning for this diagnostic.
